In [1]:
import numpy as np
from pathlib import Path
import pysacsnow as pss
from pysacsnow import Model

## Multi-model simulation
The "Model" class in pysacsnow was created to help run multiple models together in series. This is a pretty typical situation with the NWS Models i.e., a typical workflow would be to run the snow model (snow-17), then pass the snowmelt plus rain to the soil moisture accounting model (sac-sma), and then run the unit hydrolograph model to get discharge volumes from runoff depths passed from the soil moisture model. 

You could run each model individually and then pass information back and forth to the different models manually using the standard Process classes (as discussed in example 02). However, this is a helper class to make that easier.

### Multi-model configuration input files
There are 2 additional configuration files that are used to set up the multi-model run. These are:
1. `<model_name>.control.yaml` - This file contains the standard control configuration for the model simulation. This is similar to the ones used previously for the 
                             individual models, except this one only has the `simulation` inputs.
2. `<model_name>.run.yaml` - This file is new. This contains the workflow configuration for the model simulation which includes the `run order` 
                             (i.e. which model runs first, second, etc.); the `model_processes`  (i.e. which models are going to be run and where are the parameter and configuration files for those models; these files are indentical to the ones used in examples 01 and 02); and the `exchanges`. The exchanges define which (if any) variables are exchanged between the different models and how the map to one another. I.e., the input `hru_ppt` (precipitation) for the sac model would be replaced with the `raim` (rain + melt) from snow-17. 

For a typical run with snow-17, sacsma, uhg, the example run file would not need to be changed except for the paths to the configuration files in the `model_processes`

In [2]:
# Example run file
study_dir = Path("/home/kvanwerkhoven/projects/pysac/hawaii")
run_file = Path(study_dir, "configurations/multi-model/model.run.yaml")
model_dict = Model.model_dict_from_yaml(run_file)
print(model_dict)

{'control': <pysacsnow.base.control.Controller object at 0x7fbb7b269400>, 'model_order': ['nwssacsma', 'nwsuhg'], 'nwssacsma': {'class': <class 'pysacsnow.sac.sacramento_sma.SACSMA'>, 'params': <pysacsnow.base.parameters.Parameter object at 0x7fbb6a2f3410>, 'config_file': ConfigFile(simulation=Simulation(name='16103000', output_path=PosixPath('../data/output/output_sac_16103000.csv'), start_date=datetime.datetime(2021, 4, 20, 1, 0), end_date=datetime.datetime(2026, 4, 12, 0, 0), time_step=3600, time_units='s'), model=ModelConfig(name='sac', dimensions=Dim(type='nhru', size=1), parameter_type='json', parameters=Parameter(path=PosixPath('../configurations/sac/sac_params_16103000.json')), forcing_type='csv', forcing=[Forcing(name='16103000', path=PosixPath('../data/forcing/forcing.sac.16103000.csv'))], initial_conditions=InitialConditions(restart=False, path=PosixPath('restart.nc')), output_variables=None))}, 'nwsuhg': {'class': <class 'pysacsnow.unithg.uhg.UHG'>, 'params': <pysacsnow.bas

In [3]:
# Set up the model simulation based on the run file dictionary
# You could also manually create the model_dict if needed instead of loading it from the yaml file
model_sim = Model(process_model_dict=model_dict)

Output file exists - Removing/Overwritting...


In [4]:
# Run the model simulation for 1 time step
#model_sim.run(finalize=False, n_time_steps=100)

# Display the results of the simulation for the specified process (e.g., "nwssnow17") and variable (e.g., "we")
# The processes available can be found in the model dictionary. These will math the process names in the yaml file.
# The variable names can be found in the process documentation, this will interact exactly the same as the process example from notebook 02.
#display(model_sim.processes["nwssacsma"])
#display(model_sim.processes["nwssacsma"].we)

In [5]:
# The default method is to just run the model for all of the timesteps defined in the control yaml file.
model_sim.run()

  0%|          | 0/43633 [00:00<?, ?it/s]

'hru_aesc' not in input csv file; setting to NaN, you may explicitly overwrite
model.run(): finalizing
WRITE TO DISK
WRITE TO DISK
Output path exists: Appending to existing file
